<a href="https://colab.research.google.com/github/ajasrotiabe/imgprocessing/blob/main/assignment11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Install required libraries
!pip install -q tensorflow scikit-learn matplotlib pillow

import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Download Tiny ImageNet
!wget -q http://cs231n.stanford.edu/tiny-imagenet-200.zip

# Extract dataset
with zipfile.ZipFile("tiny-imagenet-200.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/")

dataset_path = "/content/tiny-imagenet-200/train"

# Select any 3 classes
classes = sorted([
    d for d in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, d))
])[:3]

print("Selected classes:", classes)

# Load 200 images from each class
X = []
y = []

for label, class_name in enumerate(classes):

    image_folder = os.path.join(
        dataset_path, class_name, "images"
    )

    image_files = sorted(os.listdir(image_folder))[:200]

    for file in image_files:
        path = os.path.join(image_folder, file)

        image = Image.open(path).convert("RGB")
        image = image.resize((64, 64))

        X.append(np.array(image))
        y.append(label)

X = np.array(X, dtype=np.float32) / 255.0
y = np.array(y)

print("Dataset shape:", X.shape)
print("Labels shape:", y.shape)

# Split: 70% training, 15% validation, 15% testing
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50,
    random_state=42, stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("Class names:", classes)


Selected classes: ['n01443537', 'n01629819', 'n01641577']
Dataset shape: (600, 64, 64, 3)
Labels shape: (600,)
Training: (420, 64, 64, 3)
Validation: (90, 64, 64, 3)
Testing: (90, 64, 64, 3)
Class names: ['n01443537', 'n01629819', 'n01641577']


In [2]:

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, Model

# 1. Initialize random 3x3 matrix
matrix = np.random.rand(3, 3)

print("Original 3x3 Matrix:\n", matrix)

# 2. Convolution using a random 3x3 kernel
kernel = np.random.rand(3, 3, 1, 1)

input_matrix = matrix.reshape(1, 3, 3, 1).astype(np.float32)

conv = tf.nn.conv2d(
    input_matrix,
    kernel.astype(np.float32),
    strides=1,
    padding='SAME'
)

print("\nConvolution with Padding P=1:\n", conv.numpy()[0, :, :, 0])

# 3. Activation functions
relu = tf.nn.relu(conv)
sigmoid = tf.nn.sigmoid(conv)

print("\nReLU:\n", relu.numpy()[0, :, :, 0])
print("\nSigmoid:\n", sigmoid.numpy()[0, :, :, 0])

# 4. Max and Min Pooling
max_pool = tf.nn.max_pool2d(
    relu, ksize=2, strides=1, padding='VALID'
)

min_pool = -tf.nn.max_pool2d(
    -relu, ksize=2, strides=1, padding='VALID'
)

print("\nMax Pooling:\n", max_pool.numpy())
print("\nMin Pooling:\n", min_pool.numpy())

# 5. Global Average Pooling
gap = layers.GlobalAveragePooling2D()(relu)

print("\nGlobal Average Pooling:\n", gap.numpy())

# 6. Build a demonstration CNN with remaining components
inputs = layers.Input(shape=(32, 32, 1))

x = layers.Conv2D(8, (3, 3), padding='same')(inputs)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)

max_features = layers.MaxPooling2D((2, 2))(x)

# Min pooling
min_features = -layers.MaxPooling2D((2, 2))(-x)

# Global Average Pooling
gap_features = layers.GlobalAveragePooling2D()(x)

# Flatten
flat = layers.Flatten()(max_features)
min_flat = layers.Flatten()(min_features)

combined = layers.Concatenate()([
    flat, min_flat, gap_features
])

# Dense layer
x = layers.Dense(32, activation='relu')(combined)

# Dropout
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(3, activation='softmax')(x)

model = Model(inputs, outputs)

model.summary()

# 7. Demonstrate Early Stopping
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

dummy_X = np.random.rand(30, 32, 32, 1).astype(np.float32)
dummy_y = np.random.randint(0, 3, 30)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    dummy_X, dummy_y,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print("\nCNN Basics implementation completed!")


Original 3x3 Matrix:
 [[0.35228756 0.57291932 0.92162213]
 [0.46917291 0.85882702 0.90465422]
 [0.3439735  0.22153786 0.50602253]]

Convolution with Padding P=1:
 [[1.624193   2.6216044  1.4429656 ]
 [1.6514963  2.5973809  1.6873642 ]
 [0.7122404  1.3646392  0.88142157]]

ReLU:
 [[1.624193   2.6216044  1.4429656 ]
 [1.6514963  2.5973809  1.6873642 ]
 [0.7122404  1.3646392  0.88142157]]

Sigmoid:
 [[0.83537257 0.9322391  0.80891347]
 [0.8390932  0.93069285 0.8438772 ]
 [0.67089605 0.79651266 0.7071167 ]]

Max Pooling:
 [[[[2.6216044]
   [2.6216044]]

  [[2.5973809]
   [2.5973809]]]]

Min Pooling:
 [[[[1.624193  ]
   [1.4429656 ]]

  [[0.7122404 ]
   [0.88142157]]]]

Global Average Pooling:
 [[1.6203673]]


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32, 8) │         80 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32, 8) │         32 │ conv2d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 32, 32, 8) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ negative (Negative) │ (None, 32, 32, 8) │          0 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 16, 16, 8) │          0 │ negative[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 16, 16, 8) │          0 │ activation[0][0]  │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ negative_1          │ (None, 16, 16, 8) │          0 │ max_pooling2d_1[… │
│ (Negative)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 2048)      │          0 │ max_pooling2d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 2048)      │          0 │ negative_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 8)         │          0 │ activation[0][0]  │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 4104)      │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0],  │
│                     │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │    131,360 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 3)         │         99 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 131,571 (513.95 KB)

 Trainable params: 131,555 (513.89 KB)

 Non-trainable params: 16 (64.00 B)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 697ms/step - accuracy: 0.3333 - loss: 2.0022 - val_accuracy: 0.0000e+00 - val_loss: 1.1280
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 264ms/step - accuracy: 0.3333 - loss: 1.7989 - val_accuracy: 0.5000 - val_loss: 1.1285
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 227ms/step - accuracy: 0.4583 - loss: 1.2086 - val_accuracy: 0.5000 - val_loss: 1.1764

CNN Basics implementation completed!


In [ ]:

import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models

# Build Custom CNN
model = models.Sequential([

    layers.Input(shape=(64, 64, 3)),

    # First Convolution Block
    layers.Conv2D(32, (3, 3), padding='same',
                  activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Second Convolution Block
    layers.Conv2D(64, (3, 3), padding='same',
                  activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Third Convolution Block
    layers.Conv2D(128, (3, 3), padding='same',
                  activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Flatten
    layers.Flatten(),

    # Dense layers
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(3, activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Early stopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

# Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

# Evaluate model
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("\nCustom CNN Test Accuracy:", test_accuracy * 100, "%")

# Plot Training and Validation Loss
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title("Training vs Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid()

# Plot Training and Validation Accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

# Predict test data
y_pred = np.argmax(model.predict(X_test), axis=1)

print("\nCustom CNN Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='macro', zero_division=0))
print("Recall:", recall_score(y_test, y_pred, average='macro', zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, average='macro', zero_division=0))

# Save model for comparison in Task 3
model.save("custom_cnn.keras")


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,143,235 (4.36 MB)

 Trainable params: 1,142,787 (4.36 MB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/20
13/14 ━━━━━━━━━━━━━━━━━━━━ 0s 909ms/step - accuracy: 0.4861 - loss: 3.0841

In [ ]:

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, MobileNetV2, ResNet50
from sklearn.metrics import accuracy_score, precision_score
from sklearn.metrics import recall_score, f1_score

# Store results
results = {}

# Function to calculate performance metrics
def evaluate_model(model, X_test, y_test, name):

    predictions = model.predict(X_test, verbose=0)
    y_pred = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro',
                                zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro',
                          zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro',
                  zero_division=0)

    results[name] = [accuracy, precision, recall, f1]

    print("\n", name)
    print("--------------------------")
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)


# Evaluate Custom CNN
custom_model = tf.keras.models.load_model("custom_cnn.keras")

evaluate_model(custom_model, X_test, y_test, "Custom CNN")


# Function to build transfer learning model
def build_transfer_model(base_model, name):

    base_model.trainable = False

    inputs = layers.Input(shape=(64, 64, 3))

    x = base_model(inputs, training=False)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(3, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print("\n", name, "Model Summary")
    model.summary()

    return model


# Function to train and evaluate transfer models
def train_transfer_model(base_model, name, preprocess):

    # Convert images from [0,1] to [0,255]
    X_train_pre = preprocess(X_train * 255.0)
    X_val_pre = preprocess(X_val * 255.0)
    X_test_pre = preprocess(X_test * 255.0)

    model = build_transfer_model(base_model, name)

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    model.fit(
        X_train_pre, y_train,
        validation_data=(X_val_pre, y_val),
        epochs=10,
        batch_size=16,
        callbacks=[early_stop],
        verbose=1
    )

    evaluate_model(model, X_test_pre, y_test, name)

    return model


# 1. VGG16
vgg_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(64, 64, 3)
)

vgg_model = train_transfer_model(
    vgg_base,
    "VGG16",
    tf.keras.applications.vgg16.preprocess_input
)


# 2. MobileNetV2
mobile_base = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(64, 64, 3)
)

mobile_model = train_transfer_model(
    mobile_base,
    "MobileNetV2",
    tf.keras.applications.mobilenet_v2.preprocess_input
)


# 3. ResNet50
resnet_base = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(64, 64, 3)
)

resnet_model = train_transfer_model(
    resnet_base,
    "ResNet50",
    tf.keras.applications.resnet50.preprocess_input
)


# Display final comparison
print("\nFINAL MODEL COMPARISON")
print("==========================================")

for name, metrics in results.items():

    print("\nModel:", name)
    print("Accuracy:", metrics[0])
    print("Precision:", metrics[1])
    print("Recall:", metrics[2])
    print("F1 Score:", metrics[3])


# Plot comparison graph
model_names = list(results.keys())

accuracies = [results[m][0] for m in model_names]
precisions = [results[m][1] for m in model_names]
recalls = [results[m][2] for m in model_names]
f1_scores = [results[m][3] for m in model_names]

x = np.arange(len(model_names))
width = 0.2

plt.figure(figsize=(12, 6))

plt.bar(x - 1.5*width, accuracies, width, label='Accuracy')
plt.bar(x - 0.5*width, precisions, width, label='Precision')
plt.bar(x + 0.5*width, recalls, width, label='Recall')
plt.bar(x + 1.5*width, f1_scores, width, label='F1 Score')

plt.xlabel("Models")
plt.ylabel("Performance Score")
plt.title("Custom CNN vs Transfer Learning Models")

plt.xticks(x, model_names, rotation=15)
plt.ylim(0, 1)
plt.legend()
plt.grid(axis='y')

plt.tight_layout()
plt.show()
